# Order Trend Validation

**Owner:** Sweta  
**Assigned reviewer:** Omar Leopoldo  
**Run after:** `03_gold_eda.ipynb`

Validates chronological months and reconciles monthly order and revenue totals with orders_gold and kpi_gold.

This notebook is an owner-specific PySpark contribution. The owner must run it personally, inspect the displayed result, understand every assertion, and commit it from their own GitHub account.


## 1. Load the validated project tables


In [ ]:
from pyspark.sql import functions as F

CATALOG = "workspace"
SCHEMA = "analytics"
spark.sql(f"USE CATALOG {CATALOG}")
spark.sql(f"USE SCHEMA {SCHEMA}")

orders_silver = spark.table(f"{CATALOG}.{SCHEMA}.orders_silver")
orders_gold = spark.table(f"{CATALOG}.{SCHEMA}.orders_gold")
monthly_revenue = spark.table(f"{CATALOG}.{SCHEMA}.monthly_revenue_gold")
kpi = spark.table(f"{CATALOG}.{SCHEMA}.kpi_gold")

print(f"orders_silver: {orders_silver.count():,}")
print(f"orders_gold: {orders_gold.count():,}")
print(f"monthly_revenue_gold: {monthly_revenue.count():,}")


## 2. Run owner-specific reconciliation and integrity checks


In [ ]:
# Gold must contain exactly one row per completed analytical order.
assert orders_gold.select("OrderID").distinct().count() == orders_gold.count()
assert orders_gold.filter(F.col("OrderTimestamp").isNull()).count() == 0
assert orders_gold.filter(F.col("YearMonth").isNull()).count() == 0

# Confirm that the monthly labels are unique and chronologically ordered.
months = [
    row["YearMonth"]
    for row in monthly_revenue.orderBy("YearMonth").select("YearMonth").collect()
]
assert months == sorted(months)
assert len(months) == len(set(months))

# Monthly totals must reconcile with one-row-per-order Gold data.
monthly_orders = monthly_revenue.agg(F.sum("CompletedOrders")).first()[0]
monthly_net_revenue = monthly_revenue.agg(F.round(F.sum("NetRevenue"), 2)).first()[0]
gold_net_revenue = orders_gold.agg(F.round(F.sum("NetRevenue"), 2)).first()[0]
published = kpi.first()

assert monthly_orders == orders_gold.count()
assert monthly_orders == published["CompletedOrders"]
assert abs(monthly_net_revenue - gold_net_revenue) < 0.01
assert abs(monthly_net_revenue - published["TotalNetRevenue"]) < 0.01

# Gold is a valid subset of cleaned orders with usable completed-order line items.
silver_completed = orders_silver.filter(F.col("OrderStatus") == "Completed").count()
assert silver_completed >= orders_gold.count()


## 3. Display the observed business result and success marker


In [ ]:
weekend_comparison = orders_gold.groupBy(
    F.when(F.col("IsWeekend"), "Weekend").otherwise("Weekday").alias("DayType")
).agg(
    F.countDistinct("OrderID").alias("CompletedOrders"),
    F.round(F.sum("NetRevenue"), 2).alias("NetRevenue"),
    F.round(F.avg("NetRevenue"), 2).alias("AverageOrderValue"),
)

print(f"Monthly periods validated: {len(months)}")
print(f"Completed orders: {monthly_orders:,}")
print(f"Net revenue: ${monthly_net_revenue:,.2f}")
display(monthly_revenue.orderBy("YearMonth"))
display(weekend_comparison.orderBy("DayType"))

print("SWETA_ORDER_TREND_VALIDATION_PASSED")


## What the owner must be able to explain

- Which tables were compared and why.
- What each assertion protects against.
- What the displayed result means for FreshRoute.
- Why the final success marker `SWETA_ORDER_TREND_VALIDATION_PASSED` only prints after every check passes.
